In [1]:
import requests
import pandas as pd
import numpy as np
import cvxpy as cp
import ecos as ecos

In [2]:
FMP_API_KEY = "ksnxCfuMd8YcYtNVDrqZKBY0aZeDMLX8"

In [3]:
def get_stock_list():
    """
    Fetches all stocks and indices with a market cap > $100M from FMP.
    Returns a list of tickers.
    """
    url = f"https://financialmodelingprep.com/api/v3/stock-screener?marketCapMoreThan=100000000&apikey={FMP_API_KEY}"
    response = requests.get(url)
    data = response.json()

    tickers = [stock["symbol"] for stock in data if "symbol" in stock]
    return tickers

In [84]:
def get_historical_prices(tickers, period="10y"):
    """
    Fetches historical price data for given tickers.
    Returns a DataFrame with dates as index and tickers as columns, sorted in ascending order.
    """
    price_data = {}
    
    for ticker in tickers:
        url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}?serietype=line&apikey={FMP_API_KEY}"
        response = requests.get(url)
        data = response.json()
        
        if "historical" in data:
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df.set_index("date", inplace=True)
            price_data[ticker] = df["close"]
    
    # Create a DataFrame and ensure it is sorted by date in ascending order
    df_prices = pd.DataFrame(price_data).sort_index(ascending=True)
    
    return df_prices

In [96]:
def compute_metrics(price_df):
    """
    Computes 1Y CAGR, 5Y CAGR, 1Y SD, and 5Y SD for each stock.
    Prints debugging information to ensure correct dates and prices are used.
    Returns a DataFrame with tickers as index and metrics as columns.
    """
    metrics = {}

    for ticker in price_df.columns:
        prices = price_df[ticker].dropna()  # Keep prices in descending order (latest first)
        
        if len(prices) < 252:  # Not enough data for 1 year
            continue

        # ✅ Fix: Use most recent dates
        end_1y = prices.index[-1]  # Most recent available date
        start_1y = end_1y - pd.DateOffset(years=1)
        start_1y = prices.index.asof(start_1y)  # Find closest valid date

        # PRINT DEBUGGING
        #print(f"{ticker} 1Y Start: {start_1y}, Price: {prices.loc[start_1y]}")
        #print(f"{ticker} 1Y End: {end_1y}, Price: {prices.loc[end_1y]}")

        # Compute 1Y CAGR and SD
        cagr_1y = (prices.loc[end_1y] / prices.loc[start_1y]) ** (1/1) - 1
        sd_1y = np.std(prices.pct_change().dropna()) * np.sqrt(252)

        if len(prices) >= 252 * 5:  # Check for 5-year data
            # ✅ Fix: Use most recent 5Y start and end
            end_5y = prices.index[-1]  # Most recent available date
            start_5y = end_5y - pd.DateOffset(years=5)
            start_5y = prices.index.asof(start_5y)  # Find closest valid date

            # PRINT DEBUGGING
            #print(f"{ticker} 5Y Start: {start_5y}, Price: {prices.loc[start_5y]}")
            #print(f"{ticker} 5Y End: {end_5y}, Price: {prices.loc[end_5y]}")

            # Compute 5Y CAGR and SD
            cagr_5y = (prices.loc[end_5y] / prices.loc[start_5y]) ** (1/5) - 1
            sd_5y = np.std(prices.pct_change().dropna()) * np.sqrt(252)
        else:
            cagr_5y, sd_5y = np.nan, np.nan

        metrics[ticker] = [cagr_1y, cagr_5y, sd_1y, sd_5y]

    return pd.DataFrame.from_dict(metrics, orient="index", columns=["1Y CAGR", "5Y CAGR", "1Y SD", "5Y SD"])

In [116]:
def optimize_portfolio(metrics_df, target_return=0.1, use_5y=True, allow_short=False, max_weight=0.3, num_stocks=10):
    """
    Optimizes portfolio allocation using Mean-Variance Optimization (MVO).
    
    Parameters:
    - target_return: Desired portfolio return (e.g., 10% = 0.1)
    - use_5y: If True, uses 5Y metrics, otherwise uses 1Y.
    - allow_short: If True, allows short selling (negative weights).
    - max_weight: Maximum allocation in a single stock.
    - num_stocks: Maximum number of stocks in the portfolio (not forced).

    Returns:
    - Optimal portfolio tickers, weights, expected return, and variance.
    """
    # Select the correct data columns
    expected_returns = metrics_df["5Y CAGR"] if use_5y else metrics_df["1Y CAGR"]
    risk_matrix = metrics_df["5Y SD"] if use_5y else metrics_df["1Y SD"]

    # Drop NaN values
    valid_stocks = expected_returns.dropna().index.intersection(risk_matrix.dropna().index)
    expected_returns = expected_returns.loc[valid_stocks]
    risk_matrix = risk_matrix.loc[valid_stocks]

    # Rank stocks by risk-adjusted return (CAGR / SD) and select top candidates
    risk_adjusted_return = expected_returns / risk_matrix
    candidate_stocks = risk_adjusted_return.nlargest(num_stocks * 2).index  # Start with a larger pool

    # Fetch historical prices for only selected stocks
    returns_df = get_historical_prices(candidate_stocks)
    
    # Compute log returns and covariance matrix
    log_returns = np.log(returns_df / returns_df.shift(1)).dropna()
    cov_matrix = log_returns.cov()

    # Ensure expected_returns and covariance matrix match in shape
    selected_stocks = expected_returns.loc[candidate_stocks].index.intersection(cov_matrix.index)
    expected_returns = expected_returns.loc[selected_stocks]
    cov_matrix = cov_matrix.loc[selected_stocks, selected_stocks]

    n = len(selected_stocks)
    if n == 0:
        return "No feasible stocks found."

    weights = cp.Variable(n)
    z = cp.Variable(n, boolean=True)  # Binary variable to track active stocks

    # Objective: Minimize portfolio variance
    portfolio_variance = cp.quad_form(weights, cov_matrix)
    objective = cp.Minimize(portfolio_variance)

    # Constraints
    constraints = [
        cp.sum(weights) == 1,  # Weights sum to 1
        expected_returns.to_numpy() @ weights >= target_return  # Target return
    ]
    
    if not allow_short:
        constraints.append(weights >= 0)  # No short selling
    constraints.append(weights <= max_weight * z)  # Enforce max weight per stock
    constraints.append(cp.sum(z) <= num_stocks)  # Limit active stocks to num_stocks

    # Solve the optimization problem
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.MOSEK if cp.MOSEK in cp.installed_solvers() else cp.ECOS_BB)

    # Get results
    if prob.status not in ["optimal", "optimal_inaccurate"]:
        return "No feasible solution found."

    optimal_weights = weights.value
    selected_stocks = selected_stocks[optimal_weights > 1e-4]  # Filter nonzero weights

    final_weights = {ticker: weight for ticker, weight in zip(selected_stocks, optimal_weights) if weight > 1e-4}
    expected_portfolio_return = expected_returns.to_numpy() @ optimal_weights
    daily_portfolio_variance = optimal_weights.T @ cov_matrix.to_numpy() @ optimal_weights
    annualized_portfolio_variance = daily_portfolio_variance * 252
    annualized_portfolio_std = np.sqrt(annualized_portfolio_variance)

    return {
    "tickers": list(selected_stocks),
    "weights": {
        ticker: float(round(weight * 100, 2)) 
        for ticker, weight in final_weights.items()
    },
    "expected_return": float(round(expected_portfolio_return * 100, 2)),
    "portfolio_std": float(round(annualized_portfolio_std * 100, 2))
}


In [127]:
tickers = get_stock_list()
price_df = get_historical_prices(tickers)
metrics = compute_metrics(price_df)

KeyboardInterrupt: 

In [124]:
optimized_portfolio = optimize_portfolio(metrics, target_return=0.1, use_5y=True, allow_short=False, max_weight=.4,num_stocks=3)

In [125]:
optimized_portfolio

{'tickers': ['SPY', 'QQQ', 'AAPL'],
 'weights': {'SPY': 40.0, 'QQQ': 40.0, 'AAPL': 20.0},
 'expected_return': 18.9,
 'portfolio_std': 23.78}